## Data Preparation

This particular notebook is dedicated to preparing the raw data in the .zip files downloaded from the Census data source. The raw data files contain fixed-width text and we will use Dec YTD General Imports customs value to get data for each calendar year. 

Name: Vighnesh Laxman Hemnani

NetID: vh2307

In [5]:
import csv
from collections import Counter
from pathlib import Path
from zipfile import ZipFile

years = range(2019, 2026)
products = {b"854231", b"854232", b"854233", b"854239", b"854290"}
raw = Path("data/raw/census")
output = Path("data/process-data.csv")

In [6]:
# function to get data per .zip file / year: 

def read_year(year):

    path = raw / f"IMDB{year % 100:02d}12.ZIP"
    period = f"{year}12".encode("ascii")
    detail = Counter()
    national = Counter()

    with ZipFile(path) as archive:
        members = {name.upper(): name for name in archive.namelist()}

        # get list of all countries: 

        with archive.open(members["COUNTRY.TXT"]) as source:
            names = {
                row[:4]: row[11:61].decode("latin-1").strip()
                for row in source
            }

        # note that censeus positions start at 1 

        with archive.open(members["IMP_DETL.TXT"]) as source:
            for row in source:
                if row[:6] in products:
                    assert row[22:28] == period, (year, "detail period")
                    detail[row[:6], row[10:14]] += int(row[508:523])

        with archive.open(members["IMP_COMM.TXT"]) as source:
            for row in source:
                if row[:6] in products:
                    assert row[66:72] == period, (year, "national period")
                    national[row[:6]] += int(row[552:567])

    detail_totals = Counter()
    for (product, origin), value in detail.items():
        assert names.get(origin) and value >= 0, (year, product, origin)
        detail_totals[product] += value

    assert set(national) == products, (year, "missing product")
    assert detail_totals == national, (year, "totals do not reconcile")

    return [
        (year, product.decode(), origin.decode(), names[origin], value)
        for (product, origin), value in sorted(detail.items())
    ]


rows = []
for year in years:
    year_rows = read_year(year)
    rows.extend(year_rows)
    print(year, len(year_rows), "rows; all 5 product totals matched")

assert len({row[:3] for row in rows}) == len(rows)

with output.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["year", "product_code", "origin_code", "origin", "import_value_usd"])
    writer.writerows(rows)

print(f"Wrote {len(rows):,} rows to {output}")

2019 340 rows; all 5 product totals matched
2020 348 rows; all 5 product totals matched
2021 374 rows; all 5 product totals matched
2022 391 rows; all 5 product totals matched
2023 358 rows; all 5 product totals matched
2024 341 rows; all 5 product totals matched
2025 329 rows; all 5 product totals matched
Wrote 2,481 rows to data/process-data.csv
